# Google Earth Engine - Interactive Satellite Visualization

This notebook provides an interactive environment for:
- Collecting Sentinel-2 satellite imagery
- Visualizing satellite data on interactive maps
- Calculating vegetation and water indices
- Exporting data for GAN+VLM training

## Prerequisites
1. Run `scripts/01_authenticate.py` first
2. Configure your `.env` file with GEE project ID
3. Install geemap: `pip install geemap`

## 1. Setup and Initialization

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import ee
import geemap
import numpy as np
from config.settings import *
from scripts.gee_utils import (
    initialize_gee, create_aoi_from_bounds, mask_sentinel2_clouds,
    calculate_ndvi, calculate_ndbi, calculate_ndwi,
    filter_collection_by_cloud_cover, filter_collection_by_date
)

print("✓ Imports successful")

In [ ]:
# Initialize Google Earth Engine
print("Initializing GEE...")
success = initialize_gee()
if success:
    print("✓ GEE initialized successfully")
else:
    print("✗ Initialization failed. Please run scripts/01_authenticate.py first.")

## 2. Define Area of Interest

In [ ]:
# Create Area of Interest from configuration
aoi = create_aoi_from_bounds(AOI_BOUNDS)

print(f"Area of Interest:")
print(f"  Longitude: {AOI_BOUNDS['min_longitude']} to {AOI_BOUNDS['max_longitude']}")
print(f"  Latitude: {AOI_BOUNDS['min_latitude']} to {AOI_BOUNDS['max_latitude']}")

# Calculate center for map
center_lat = (AOI_BOUNDS['min_latitude'] + AOI_BOUNDS['max_latitude']) / 2
center_lon = (AOI_BOUNDS['min_longitude'] + AOI_BOUNDS['max_longitude']) / 2
print(f"\nCenter coordinates: ({center_lat}, {center_lon})")

## 3. Collect Sentinel-2 Images

In [ ]:
# Collect Sentinel-2 images
print(f"Collecting Sentinel-2 images...")
print(f"  Dataset: {SATELLITE_DATASET}")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Cloud cover threshold: {CLOUD_COVER_THRESHOLD}%")

collection = ee.ImageCollection(SATELLITE_DATASET) \
    .filterBounds(aoi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_COVER_THRESHOLD)) \
    .map(mask_sentinel2_clouds) \
    .limit(MAX_IMAGES)

size = collection.size().getInfo()
print(f"\n✓ Retrieved {size} images")

if size > 0:
    first_image = ee.Image(collection.first())
    info = first_image.getInfo()
    print(f"  Bands available: {len(info['bands'])}")

## 4. Create Median Composite and Calculate Indices

In [ ]:
# Create median composite
print("Creating median composite...")
median_image = collection.median().clip(aoi)

# Calculate indices
print("Calculating indices...")
ndvi = calculate_ndvi(median_image)
ndbi = calculate_ndbi(median_image)
ndwi = calculate_ndwi(median_image)

print("✓ Composite and indices created")

## 5. Interactive Map Visualization - RGB

In [ ]:
# Create interactive map for RGB visualization
Map1 = geemap.Map(center=(center_lat, center_lon), zoom=8)

# RGB visualization parameters
rgb_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3,
    'gamma': 1.2
}

# Add RGB layer
Map1.addLayer(median_image, rgb_params, 'Sentinel-2 RGB')
Map1.addLayerControl()

print("✓ RGB map created")
Map1

## 6. Interactive Map - Vegetation Index (NDVI)

In [ ]:
# Create map for NDVI
Map2 = geemap.Map(center=(center_lat, center_lon), zoom=8)

# NDVI visualization
ndvi_params = {
    'min': -1,
    'max': 1,
    'palette': ['red', 'yellow', 'green']
}

Map2.addLayer(median_image, rgb_params, 'RGB', shown=False)
Map2.addLayer(ndvi, ndvi_params, 'NDVI (Vegetation)')
Map2.addLayerControl()

print("✓ NDVI map created")
Map2

## 7. Interactive Map - Built-up Index (NDBI)

In [ ]:
# Create map for NDBI
Map3 = geemap.Map(center=(center_lat, center_lon), zoom=8)

# NDBI visualization
ndbi_params = {
    'min': -1,
    'max': 1,
    'palette': ['blue', 'white', 'red']
}

Map3.addLayer(median_image, rgb_params, 'RGB', shown=False)
Map3.addLayer(ndbi, ndbi_params, 'NDBI (Built-up)')
Map3.addLayerControl()

print("✓ NDBI map created")
Map3

## 8. Interactive Map - Water Index (NDWI)

In [ ]:
# Create map for NDWI
Map4 = geemap.Map(center=(center_lat, center_lon), zoom=8)

# NDWI visualization
ndwi_params = {
    'min': -1,
    'max': 1,
    'palette': ['red', 'white', 'blue']
}

Map4.addLayer(median_image, rgb_params, 'RGB', shown=False)
Map4.addLayer(ndwi, ndwi_params, 'NDWI (Water)')
Map4.addLayerControl()

print("✓ NDWI map created")
Map4

## 9. Image Statistics

In [ ]:
# Calculate statistics for the AOI
print("Calculating statistics...\n")

combined = median_image.select(['B2', 'B3', 'B4', 'B8', 'B11']).addBands(ndvi).addBands(ndbi).addBands(ndwi)

stats = combined.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=aoi,
    scale=EXPORT_SCALE,
    maxPixels=int(MAX_PIXELS)
)

stats_dict = stats.getInfo()

print("Band Statistics:")
print("-" * 50)
for key in sorted(stats_dict.keys()):
    if 'mean' in key:
        band = key.replace('_mean', '')
        min_key = f"{band}_min"
        max_key = f"{band}_max"
        min_val = stats_dict.get(min_key, 0)
        max_val = stats_dict.get(max_key, 0)
        mean_val = stats_dict[key]
        print(f"{band:>5}: mean={mean_val:7.4f}  min={min_val:7.4f}  max={max_val:7.4f}")
print("-" * 50)

## 10. Export Data

In [ ]:
# Export RGB composite to Google Drive
from scripts.gee_utils import create_export_task, start_export_task

print("Creating export tasks...\n")

# Export RGB
rgb_image = median_image.select(['B4', 'B3', 'B2']).clip(aoi)
task_rgb = create_export_task(
    image=rgb_image,
    description='sentinel2_rgb_notebook',
    folder=GOOGLE_DRIVE_FOLDER,
    aoi=aoi
)

print("Export tasks created. To start export:")
print("  task_rgb.start()")
print("\nOr uncomment the cells below to auto-start...")

In [ ]:
# Uncomment to start export automatically
# print("Starting exports...")
# start_export_task(task_rgb)
# print("\nCheck Google Drive folder: {}".format(GOOGLE_DRIVE_FOLDER))

## 11. Next Steps

### For GAN Training:
1. Download exported GeoTIFF files from Google Drive
2. Resize images to 256×256 or 128×128 pixels
3. Organize by category: `flood/`, `urban/`, `forest/`, etc.
4. Use for GAN training or style transfer

### For VLM Fine-tuning:
1. Prepare image+caption pairs
2. Create captions: `"flood agricultural area"`, `"urban settlement"`, etc.
3. Use for vision-language model fine-tuning

### For Further Processing:
1. Use `scripts/04_process_images.py` for batch processing
2. Use `scripts/06_export_to_drive.py` for automated exports
3. Adjust parameters in `config/settings.py` or `.env` file

---

**Questions or Issues?** See README.md in the project root.